In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor


In [ ]:
df =pd.read_csv("student_habits_performance.csv")
df.head()

In [ ]:
df.info()

In [ ]:
sns.set(style="whitegrid")

In [ ]:
df.isna().sum()

In [ ]:
df =df.dropna()
df.duplicated().sum()
import warnings
warnings.filterwarnings("ignore")
df.describe(include="object").columns


In [ ]:
categorical_col=['student_id', 'gender', 'part_time_job', 'diet_quality',
       'parental_education_level', 'internet_quality',
       'extracurricular_participation']
for col in categorical_col:
  print(f"Value counts for {col}: \n {df[col].value_counts()}")

In [ ]:
df.hist(bins=20,edgecolor='black')
plt.tight_layout()
plt.show()

In [ ]:
sns.heatmap(df.corr(numeric_only=True),cmap="coolwarm",annot=True,fmt='.2f')
plt.title("Correlation matrix")
plt.show()

In [ ]:
df.describe().columns

In [ ]:
num_features = ['age', 'study_hours_per_day', 'social_media_hours', 'netflix_hours',
       'attendance_percentage', 'sleep_hours', 'exercise_frequency',
       'mental_health_rating', ]
for feature in num_features:
  sns.scatterplot(data=df,x=feature,y="exam_score")
  plt.title(f"{feature} vs Exam score")
  plt.show()

In [ ]:
for col in categorical_col:
  sns.boxplot(data=df,x=col,y="exam_score")
  plt.title(f"Exam score by {col}")
  plt.xticks(rotation=45)
  plt.show()

In [ ]:
features=['attendance_percentage','mental_health_rating','study_hours_per_day','sleep_hours','part_time_job']

In [ ]:
target="exam_score"

In [ ]:
df_model= df[features + [target]].copy()


In [ ]:
df_model

In [ ]:
le =LabelEncoder()
df_model["part_time_job"]=le.fit_transform(df_model["part_time_job"])

In [ ]:
df_model

In [ ]:
X = df_model[features]

In [ ]:
y = df_model[target]

In [ ]:
X_train ,X_test ,y_train ,y_test = train_test_split(X,y,test_size=0.2)

In [ ]:
len(y_test)

In [ ]:
len(y_train)

In [ ]:
models={
  "LinearRegression":{
    "model":LinearRegression(),
    "params":{}
  },
  "DecisionTree" : {
    "model":DecisionTreeRegressor(),
    "params":{"max_depth":[3,5,10],"min_samples_split":[2,5]}
  },
  "RandomForest":{
    "model":RandomForestRegressor(),
    "params":{"n_estimators":[50,100],"max_depth":[5,10]}
  }
}

In [ ]:
best_models =[]

In [ ]:
for name,config in models.items():
  print(f"Training {name}")
  grid =GridSearchCV(config["model"],config["params"],cv=5,scoring="neg_mean_squared_error")
  grid.fit(X_train,y_train)

  y_pred=grid.predict(X_test)
  rmse = np.sqrt(mean_squared_error(y_test,y_pred))
  r2=r2_score(y_test,y_pred)

  best_models.append({
    "model":name,
    "best_params" : grid.best_params_,
    "rmse":rmse,
    "r2":r2
  })


In [ ]:
best_models

In [ ]:
results_df = pd.DataFrame(best_models)

In [ ]:
results_df.sort_values(by="rmse")

In [ ]:
import joblib
best_row=results_df.sort_values(by="rmse").iloc[0]

In [ ]:
best_row

In [ ]:
best_model_name=best_row["model"]

In [ ]:
best_model_name

In [ ]:
best_model_config=models[best_model_name]

In [ ]:
best_model_config